In [1]:
import os
import requests
import pandas as pd
from datetime import date, datetime
import pytz

# === Get today's date in Eastern time for consistency ===
eastern = pytz.timezone('US/Eastern')
today = datetime.now(eastern).date()
today_str = today.isoformat()

def fetch_starting_pitchers(target_date):
    """
    Fetches the probable starting pitchers for each MLB game on the given date.
    """
    date_str = target_date.isoformat()
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}&hydrate=probablePitcher(note,stats,person)'

    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    pitcher_data = []
    for game in games:
        home_team = game['teams']['home']['team']['name']
        away_team = game['teams']['away']['team']['name']

        home_pitcher = game['teams']['home'].get('probablePitcher', {}).get('fullName', 'TBD')
        away_pitcher = game['teams']['away'].get('probablePitcher', {}).get('fullName', 'TBD')

        pitcher_data.append({
            'Date': date_str,
            'Away Team': away_team,
            'Away Starter': away_pitcher,
            'Home Team': home_team,
            'Home Starter': home_pitcher
        })

    return pd.DataFrame(pitcher_data)

# === MAIN ===
df = fetch_starting_pitchers(today)

if df is not None and not df.empty:
    output_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "starting-pitchers"))
    os.makedirs(output_dir, exist_ok=True)

    output_file = os.path.join(output_dir, f"starting_pitchers_{today_str}.csv")
    df.to_csv(output_file, index=False)

    print(f"✅ Starting pitchers saved to {output_file}")
    print(df.head())
else:
    print("⚠️ No starting pitcher data found.")


✅ Starting pitchers saved to /workspaces/MLB_Model/data/starting-pitchers/starting_pitchers_2025-05-30.csv
         Date           Away Team     Away Starter              Home Team  \
0  2025-05-30     Cincinnati Reds    Andrew Abbott           Chicago Cubs   
1  2025-05-30   Chicago White Sox    Jared Shuster      Baltimore Orioles   
2  2025-05-30   Milwaukee Brewers          DL Hall  Philadelphia Phillies   
3  2025-05-30           Athletics  Jeffrey Springs      Toronto Blue Jays   
4  2025-05-30  Los Angeles Angels     José Soriano    Cleveland Guardians   

     Home Starter  
0       Colin Rea  
1      Zach Eflin  
2  Taijuan Walker  
3   Chris Bassitt  
4   Luis L. Ortiz  
